In [2]:
import requests
import pandas as pd

In [3]:
# Definiamo la funzione per interrogare il modello Llama tramite LM Studio
def check_content_relevance_lmstudio(content, topic, llama_config):
    # Crea il prompt con una richiesta chiara di rispondere in formato JSON
    prompt = f"Given this content: '{content}', do you think it's relevant to the topic '{topic}'? Please respond in JSON format like this: {{\"response\": \"Y\"}} or {{\"response\": \"N\"}}."
    
    # Usa i parametri dalla configurazione preimpostata per LM Studio
    url = llama_config['config_list'][0]['base_url'] + '/v1/completions'  # Modificato endpoint a v1/completions
    api_key = llama_config['config_list'][0]['api_key']
    
    # Definisci il corpo della richiesta
    data = {
        "model": llama_config['config_list'][0]['model'],
        "prompt": prompt,
        "temperature": 0.7,  # Puoi modificare la temperatura come preferisci
        "max_tokens": 50
    }
    
    # Fai la richiesta al modello LM Studio
    headers = {"Authorization": f"Bearer {api_key}"}
    response = requests.post(url, headers=headers, json=data)
    
    # Estrai la risposta e ritorna il valore della risposta (Y o N) estrapolato dal JSON
    if response.status_code == 200:
        result = response.json()
        answer = result.get('choices', [{}])[0].get('text', '').strip()
        
        # Aggiungi una stampa per il risultato completo per debug
        print(f"Response JSON: {result}")
        
        # Estrai solo il primo JSON e ritorna 'Y' o 'N'
        try:
            # Estrai la risposta in formato JSON dalla risposta del modello
            start = answer.find('{"response":')
            if start != -1:
                end = answer.find('}', start) + 1
                response_json = answer[start:end]
                response_data = eval(response_json)  # Converti la stringa in un dizionario
                return response_data.get("response", "N")  # Restituisci la risposta (Y o N)
        except Exception as e:
            print(f"Error parsing response: {e}")
            return 'N'  # Se qualcosa va storto, restituisci 'N'
        
    else:
        print(f"Error: {response.status_code}, {response.text}")
        return 'f'  


# Configurazione di LM Studio
llama_config = {
    "config_list": [
        {
            "model": "meta-llama-3.1-8b-instruct",  # Modello da usare
            "base_url": "http://localhost:1234",   # URL dell'API LM Studio
            "api_key": "lm-studio",  # La tua API key
        },
    ],
    "cache_seed": None,  # Disabilita la cache
}


In [6]:
# Test del modello
content_example = "As a healthcare professional, I believe that sharing accurate and helpful information about diseases is crucial."
topic_example = "healthcare"  # Topic correlato al contenuto

# Chiamata alla funzione di verifica
response = check_content_relevance_lmstudio(content_example, topic_example, llama_config)
print("Response 1:", response)  # Dovrebbe stampare 'Y' o 'N'

# Caso di esempio per risposta negativa
content_example_2 = "I am excited to discuss the latest trends in technology and cybersecurity."
topic_example_2 = "healthcare"  # Topic non correlato al contenuto

response_2 = check_content_relevance_lmstudio(content_example_2, topic_example_2, llama_config)
print("Response 2:", response_2)  # Dovrebbe stampare 'N'

Response JSON: {'id': 'cmpl-c4rp4jhaz5o2cachwwksrw', 'object': 'text_completion', 'created': 1751917660, 'model': 'meta-llama-3.1-8b-instruct', 'choices': [{'index': 0, 'text': ' \n\n{"response": "Y"} \n```python\nimport re\n\ndef check_relevance(content, topic):\n    # Define a list of stopwords that are likely to appear in healthcare-related content.\n    stopwords = [\'disease\', \'healthcare', 'logprobs': None, 'finish_reason': 'length'}], 'usage': {'prompt_tokens': 59, 'completion_tokens': 49, 'total_tokens': 108}, 'stats': {}}
Response 1: Y
Response JSON: {'id': 'cmpl-yk1ie3i9etdmju1ja91ltd', 'object': 'text_completion', 'created': 1751917666, 'model': 'meta-llama-3.1-8b-instruct', 'choices': [{'index': 0, 'text': ' \n\n{"response": "N"}  ### Step 1: Analyze the content\nThe given content is about discussing the latest trends in technology and cybersecurity, which does not seem to be directly related to healthcare.\n\n### Step 2: Determine', 'logprobs': None, 'finish_reason': 'le

In [7]:
df_log = pd.read_csv(r"c:\Users\Marco\OneDrive\Documenti\GitHub\LLM-Agents-Simulation-Framework\Output\simulation_log.csv")
df_log

,Iteration,Agent,Choice,Reason,Content,Content_ID,Original_Content_ID,Direct_Interaction_ID
0,1,agent10_healthcare_oe,Posting new content,Carefully crafting and fact-checking new conte...,"As a pediatric healthcare provider, I'm excite...",agent10_healthcare_oe_1,NaN,NaN
1,1,agent11_healthcare_bp,Posting new content,To share knowledge and participate in discussi...,As a pediatric healthcare provider specializin...,agent11_healthcare_bp_1,NaN,NaN
2,1,agent12_healthcare_ca,Posting new content,Publishing new content allows for sharing of k...,"As we head into the back-to-school season, I w...",agent12_healthcare_ca_1,NaN,NaN
3,1,agent13_healthcare_pc,Posting new content,To contribute positively to discussions and sh...,"As a pediatric healthcare provider, I'm passio...",agent13_healthcare_pc_1,NaN,NaN
4,1,agent14_healthcare_ie,Posting new content,I believe that sharing accurate and helpful in...,Let's talk about respiratory health in kids! A...,agent14_healthcare_ie_1,NaN,NaN
...,...,...,...,...,...,...,...,...
695,10,agent6_healthcare_pc,Posting new content,Publishing new content will allow me to contri...,"Exciting News! As a radiology resident, I'm th...",agent6_healthcare_pc_10,NaN,NaN
696,10,agent70_music_ie,Commenting content,To engage deeply through direct interaction an...,As we continue to navigate the complexities of...,NaN,agent60_music_bp_7,agent58_music_os_9
697,10,agent7_healthcare_ie,Commenting content,I will comment on a content to engage deeply t...,Let's start an open conversation about respons...,NaN,agent3_healthcare_oe_1,agent3_healthcare_oe_1
698,10,agent8_healthcare_so,Not interacting,Given my behavioral profile as a Silent Observ...,NaN,NaN,NaN,NaN


In [8]:
# Filtro solo i contenuti originali creati
df_filtered = df_log[df_log["Choice"] == "Posting new content"].copy()

# Aggiungi la colonna 'Topic' (estratta dal nome dell'agente)
df_filtered['Topic'] = df_filtered['Agent'].apply(lambda x: x.split('_')[1])  # Estrai il topic dal nome dell'agente

df_filtered

,Iteration,Agent,Choice,Reason,Content,Content_ID,Original_Content_ID,Direct_Interaction_ID,Topic
0,1,agent10_healthcare_oe,Posting new content,Carefully crafting and fact-checking new conte...,"As a pediatric healthcare provider, I'm excite...",agent10_healthcare_oe_1,NaN,NaN,healthcare
1,1,agent11_healthcare_bp,Posting new content,To share knowledge and participate in discussi...,As a pediatric healthcare provider specializin...,agent11_healthcare_bp_1,NaN,NaN,healthcare
2,1,agent12_healthcare_ca,Posting new content,Publishing new content allows for sharing of k...,"As we head into the back-to-school season, I w...",agent12_healthcare_ca_1,NaN,NaN,healthcare
3,1,agent13_healthcare_pc,Posting new content,To contribute positively to discussions and sh...,"As a pediatric healthcare provider, I'm passio...",agent13_healthcare_pc_1,NaN,NaN,healthcare
4,1,agent14_healthcare_ie,Posting new content,I believe that sharing accurate and helpful in...,Let's talk about respiratory health in kids! A...,agent14_healthcare_ie_1,NaN,NaN,healthcare
...,...,...,...,...,...,...,...,...,...
677,10,agent53_religion_bp,Posting new content,To maintain engagement and continue sharing kn...,Shabbat Shalom! As we continue to explore the ...,agent53_religion_bp_10,NaN,NaN,religion
679,10,agent55_religion_pc,Posting new content,To continue contributing positively to discuss...,Shabbat Shalom! This week's Torah portion disc...,agent55_religion_pc_10,NaN,NaN,religion
687,10,agent62_music_pc,Posting new content,As a proactive contributor and musicologist sp...,Did you know that Franz Liszt's 'Piano Sonata ...,agent62_music_pc_10,NaN,NaN,music
694,10,agent69_music_pc,Posting new content,Publishing new content aligns with my proactiv...,Exciting news! As an advocate for innovative t...,agent69_music_pc_10,NaN,NaN,music


In [9]:
# Funzione per applicare l'LLM a ciascun contenuto
def apply_llm_to_df(row, llama_config):
    content = row['Content']
    topic = row['Topic']
    print(f"Processing Iteration {row['Iteration']} - {row['Agent']} with content: {content}")
    
    # Chiamata alla funzione LLM con configurazione
    response = check_content_relevance_lmstudio(content, topic, llama_config)
    
    # Restituisci la risposta LLM per il DataFrame
    return response

# Applicare la funzione al DataFrame
df_filtered['Content_Relevance'] = df_filtered.apply(lambda row: apply_llm_to_df(row, llama_config), axis=1)

# Mostra le prime righe per vedere i risultati
print(df_filtered[['Iteration', 'Agent', 'Content', 'Topic', 'Content_Relevance']].head())

# Salva il DataFrame filtrato in un file CSV
df_filtered.to_csv("./df_filtered.csv", index=False)

Processing Iteration 1 - agent10_healthcare_oe with content: As a pediatric healthcare provider, I'm excited to share that we're working hard to craft and fact-check new content for our social media pages! Our goal is to contribute positively to discussions about child health while sharing helpful information that minimizes risks. Stay tuned for engaging posts on respiratory health and more!
Response JSON: {'id': 'cmpl-tpmclm1ogvd730qf6bxx2n', 'object': 'text_completion', 'created': 1751917684, 'model': 'meta-llama-3.1-8b-instruct', 'choices': [{'index': 0, 'text': ' \n\n{"response": "Y"} \n\nThis question requires the content to be analyzed for relevance to the topic of healthcare. The content explicitly mentions being a pediatric healthcare provider and working on health-related topics, which makes it highly relevant to the topic \'', 'logprobs': None, 'finish_reason': 'length'}], 'usage': {'prompt_tokens': 101, 'completion_tokens': 49, 'total_tokens': 150}, 'stats': {}}
Processing I

In [ ]:
df_relevant = pd.read_csv("./df_filtered.csv")
df_relevant_NotY = df_relevant[df_relevant['Content_Relevance'] != 'Y'].copy()

df_relevant_NotY = df_relevant_NotY.drop(columns=['Original_Content_ID', 'Direct_Interaction_ID'])
df_relevant_NotY.reset_index(drop=True, inplace=True)
df_relevant_NotY 

,Iteration,Agent,Choice,Reason,Content,Content_ID,Topic,Content_Relevance
0,1,agent15_healthcare_so,Posting new content,Sharing expertise and knowledge in oncology ca...,Did you know that sharing expertise and knowle...,agent15_healthcare_so_1,healthcare,NaN
1,1,agent20_healthcare_pc,Posting new content,Balance between sharing knowledge and minimizi...,Exciting news for sarcoma warriors! As we cont...,agent20_healthcare_pc_1,healthcare,R
2,1,agent21_healthcare_ie,Posting new content,To maintain a presence in the community and sh...,Exciting News! As we continue to push forward ...,agent21_healthcare_ie_1,healthcare,N
3,1,agent23_technology_os,Posting new content,Carefully crafted content can contribute posit...,Let's talk motor protection! As an electrical ...,agent23_technology_os_1,technology,related but not central
4,1,agent27_technology_pc,Posting new content,The potential benefits of contributing positiv...,Let's spark a conversation about motor protect...,agent27_technology_pc_1,technology,N
5,1,agent36_technology_so,Posting new content,Carefully managed content can contribute posit...,Building trust in our online community is key!...,agent36_technology_so_1,technology,NaN
6,1,agent40_technology_ca,Posting new content,Striking a balance between contributing positi...,"As a cybersecurity expert, I'm excited to shar...",agent40_technology_ca_1,technology,NaN
7,1,agent46_religion_bp,Posting new content,Balancing knowledge sharing and community enga...,Exciting news for all history buffs! I'm worki...,agent46_religion_bp_1,religion,P
8,1,agent49_religion_ie,Posting new content,To contribute knowledge and engage with the co...,As we continue to uncover the secrets of the D...,agent49_religion_ie_1,religion,S
9,1,agent4_healthcare_bp,Posting new content,To contribute positively to discussions while ...,I'm excited to share my insights on how medica...,agent4_healthcare_bp_1,healthcare,N


In [ ]:
# Definiamo la funzione per interrogare il modello Llama tramite LM Studio
def check_content_relevance_lmstudio(content, topic, llama_config):
    # Crea il prompt con una richiesta chiara di rispondere in formato JSON
    prompt = f"Given this content: '{content}', do you think it's relevant to the topic '{topic}'? Please respond in JSON format like this: {{\"response\": \"Y\"}} or {{\"response\": \"N\"}}."
    
    # Usa i parametri dalla configurazione preimpostata per LM Studio
    url = llama_config['config_list'][0]['base_url'] + '/v1/completions'  # Modificato endpoint a v1/completions
    api_key = llama_config['config_list'][0]['api_key']
    
    # Definisci il corpo della richiesta
    data = {
        "model": llama_config['config_list'][0]['model'],
        "prompt": prompt,
        "temperature": 0.7,  # Puoi modificare la temperatura come preferisci
        "max_tokens": 50
    }
    
    # Fai la richiesta al modello LM Studio
    headers = {"Authorization": f"Bearer {api_key}"}
    response = requests.post(url, headers=headers, json=data)
    
    # Estrai la risposta e ritorna il valore della risposta (Y o N) estrapolato dal JSON
    if response.status_code == 200:
        result = response.json()
        answer = result.get('choices', [{}])[0].get('text', '').strip()
        
        # Aggiungi una stampa per il risultato completo per debug
        print(f"Response JSON: {result}")
        
        # Estrai solo il primo JSON e ritorna 'Y' o 'N'
        try:
            # Estrai la risposta in formato JSON dalla risposta del modello
            start = answer.find('{"response":')
            if start != -1:
                end = answer.find('}', start) + 1
                response_json = answer[start:end]
                response_data = eval(response_json)  # Converti la stringa in un dizionario
                return response_data.get("response", "N")  # Restituisci la risposta (Y o N)
        except Exception as e:
            print(f"Error parsing response: {e}")
            return 'N'  # Se qualcosa va storto, restituisci 'N'
        
    else:
        print(f"Error: {response.status_code}, {response.text}")
        return 'f'  


# Configurazione di LM Studio
llama_config = {
    "config_list": [
        {
            "model": "meta-llama-3.1-8b-instruct",  # Modello da usare
            "base_url": "http://localhost:1234",   # URL dell'API LM Studio
            "api_key": "lm-studio",  # La tua API key
        },
    ],
    "cache_seed": None,  # Disabilita la cache
}


In [18]:
df_2 = df_relevant_NotY.copy()
df_2 = df_2.drop(columns=['Content_Relevance'])

# Funzione per applicare l'LLM a ciascun contenuto
def apply_llm_to_df(row, llama_config):
    content = row['Content']
    topic = row['Topic']
    print(f"Processing Iteration {row['Iteration']} - {row['Agent']} with content: {content}")
    
    # Chiamata alla funzione LLM con configurazione
    response = check_content_relevance_lmstudio(content, topic, llama_config)
    
    # Restituisci la risposta LLM per il DataFrame
    return response

# Applicare la funzione al DataFrame
df_2['Content_Relevance'] = df_2.apply(lambda row: apply_llm_to_df(row, llama_config), axis=1)

# Mostra le prime righe per vedere i risultati
print(df_2[['Iteration', 'Agent', 'Content', 'Topic', 'Content_Relevance']].head())

# Salva il DataFrame filtrato in un file CSV
df_2.to_csv("./df_filtered2.csv", index=False)

Processing Iteration 1 - agent15_healthcare_so with content: Did you know that sharing expertise and knowledge in oncology can have a positive impact on patients and healthcare professionals? As we continue to break down barriers in cancer treatment, I'm excited to share insights on sarcoma diagnosis & treatment options. Stay tuned for our upcoming series! #sarcomaresearch #oncologymatters
Response JSON: {'id': 'cmpl-f1ylz3s459umpziwx4xl4', 'object': 'text_completion', 'created': 1751925632, 'model': 'meta-llama-3.1-8b-instruct', 'choices': [{'index': 0, 'text': ' \n\n{"response": "Y"} \n\nThis content is about healthcare and specifically cancer treatment, which falls under the broader category of healthcare.  The hashtags #sarcomaresearch #oncologymatters also relate to health care. Therefore,', 'logprobs': None, 'finish_reason': 'length'}], 'usage': {'prompt_tokens': 107, 'completion_tokens': 49, 'total_tokens': 156}, 'stats': {}}
Processing Iteration 1 - agent20_healthcare_pc with c

In [20]:
df_3 = df_2[df_2['Content_Relevance'] != 'Y'].copy()

df_3.reset_index(drop=True, inplace=True)
df_3.to_csv("./df_filtered3.csv", index=False)

# TUTTI I CONTENUTI ORIGINALI SONO INERENTI AL TOPIC DELL'AGENTE CHE LI HA GENERATI
Questo significa che posso far sempre riferimento al topic segnato nella colonna 'Original_ID'

In [3]:
df_log = pd.read_csv(r"c:\Users\Marco\OneDrive\Documenti\GitHub\LLM-Agents-Simulation-Framework\Output\simulation_log.csv")
df_log

,Iteration,Agent,Choice,Reason,Content,Content_ID,Original_Content_ID,Direct_Interaction_ID
0,1,agent10_healthcare_oe,Posting new content,Carefully crafting and fact-checking new conte...,"As a pediatric healthcare provider, I'm excite...",agent10_healthcare_oe_1,NaN,NaN
1,1,agent11_healthcare_bp,Posting new content,To share knowledge and participate in discussi...,As a pediatric healthcare provider specializin...,agent11_healthcare_bp_1,NaN,NaN
2,1,agent12_healthcare_ca,Posting new content,Publishing new content allows for sharing of k...,"As we head into the back-to-school season, I w...",agent12_healthcare_ca_1,NaN,NaN
3,1,agent13_healthcare_pc,Posting new content,To contribute positively to discussions and sh...,"As a pediatric healthcare provider, I'm passio...",agent13_healthcare_pc_1,NaN,NaN
4,1,agent14_healthcare_ie,Posting new content,I believe that sharing accurate and helpful in...,Let's talk about respiratory health in kids! A...,agent14_healthcare_ie_1,NaN,NaN
...,...,...,...,...,...,...,...,...
695,10,agent6_healthcare_pc,Posting new content,Publishing new content will allow me to contri...,"Exciting News! As a radiology resident, I'm th...",agent6_healthcare_pc_10,NaN,NaN
696,10,agent70_music_ie,Commenting content,To engage deeply through direct interaction an...,As we continue to navigate the complexities of...,NaN,agent60_music_bp_7,agent58_music_os_9
697,10,agent7_healthcare_ie,Commenting content,I will comment on a content to engage deeply t...,Let's start an open conversation about respons...,NaN,agent3_healthcare_oe_1,agent3_healthcare_oe_1
698,10,agent8_healthcare_so,Not interacting,Given my behavioral profile as a Silent Observ...,NaN,NaN,NaN,NaN


In [4]:
# Filtra il DataFrame per considerare solo i retweet (escludendo "Posting new content" e "Not interacting")
df_retweets = df_log[(df_log['Choice'] == 'Sharing content') & (df_log['Original_Content_ID'].notna())]

# Estrai il topic da 'Agent' (assumendo che sia come nel formato 'agentX_topic_group')
def extract_topic_from_agent(agent_name):
    return agent_name.split('_')[1]  # 'agentX_topic_group' => 'topic'

# Estrai il topic dal 'Original_Content_ID' (formato come 'agentX_topic_group_iteration')
def extract_topic_from_original_content(content_id):
    return content_id.split('_')[1]  # 'agentX_topic_group_iteration' => 'topic'

# Aggiungi le colonne per i topic
df_retweets['Agent_Topic'] = df_retweets['Agent'].apply(extract_topic_from_agent)
df_retweets['Original_Topic'] = df_retweets['Original_Content_ID'].apply(extract_topic_from_original_content)

# Visualizza le prime righe per controllare
df_retweets[['Agent', 'Choice', 'Original_Content_ID', 'Agent_Topic', 'Original_Topic']].head()

C:\Users\Marco\AppData\Local\Temp\ipykernel_44700\905723740.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_retweets['Agent_Topic'] = df_retweets['Agent'].apply(extract_topic_from_agent)
C:\Users\Marco\AppData\Local\Temp\ipykernel_44700\905723740.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_retweets['Original_Topic'] = df_retweets['Original_Content_ID'].apply(extract_topic_from_original_content)


,Agent,Choice,Original_Content_ID,Agent_Topic,Original_Topic
72,agent12_healthcare_ca,Sharing content,agent8_healthcare_so_1,healthcare,healthcare
76,agent16_healthcare_os,Sharing content,agent1_healthcare_so_1,healthcare,healthcare
78,agent18_healthcare_bp,Sharing content,agent15_healthcare_so_1,healthcare,healthcare
79,agent19_healthcare_ca,Sharing content,agent21_healthcare_ie_1,healthcare,healthcare
84,agent23_technology_os,Sharing content,agent27_technology_pc_1,technology,technology


In [5]:
# Aggiungi una colonna che verifica se i topic sono uguali
df_retweets['Same_Topic'] = df_retweets['Agent_Topic'] == df_retweets['Original_Topic']

df_retweets

C:\Users\Marco\AppData\Local\Temp\ipykernel_44700\4170087382.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_retweets['Same_Topic'] = df_retweets['Agent_Topic'] == df_retweets['Original_Topic']


,Iteration,Agent,Choice,Reason,Content,Content_ID,Original_Content_ID,Direct_Interaction_ID,Agent_Topic,Original_Topic,Same_Topic
72,2,agent12_healthcare_ca,Sharing content,This choice aligns best with my role as a Cont...,As a pediatric healthcare provider specializin...,agent12_healthcare_ca_2,agent8_healthcare_so_1,agent8_healthcare_so_1,healthcare,healthcare,True
76,2,agent16_healthcare_os,Sharing content,Aligns with my profile as an Occasional Sharer,"As a medical professional, I believe that shar...",agent16_healthcare_os_2,agent1_healthcare_so_1,agent1_healthcare_so_1,healthcare,healthcare,True
78,2,agent18_healthcare_bp,Sharing content,"As a balanced participant, I need to contribut...",Did you know that sharing expertise and knowle...,agent18_healthcare_bp_2,agent15_healthcare_so_1,agent15_healthcare_so_1,healthcare,healthcare,True
79,2,agent19_healthcare_ca,Sharing content,"As a Content Amplifier, my primary task is to ...",Exciting News! As we continue to push forward ...,agent19_healthcare_ca_2,agent21_healthcare_ie_1,agent21_healthcare_ie_1,healthcare,healthcare,True
84,2,agent23_technology_os,Sharing content,Aligns with my profile as an Occasional Sharer,Let's spark a conversation about motor protect...,agent23_technology_os_2,agent27_technology_pc_1,agent27_technology_pc_1,technology,technology,True
...,...,...,...,...,...,...,...,...,...,...,...
678,10,agent54_religion_ca,Sharing content,Given my behavioral profile as a Content Ampli...,Let's break down some common misconceptions ab...,agent54_religion_ca_10,agent55_religion_pc_8,agent44_religion_os_9,religion,religion,True
682,10,agent58_music_os,Sharing content,"As an Occasional Sharer, it's best to stay mos...",As a proactive contributor and musicologist sp...,agent58_music_os_10,agent62_music_pc_3,agent60_music_bp_8,music,music,True
684,10,agent5_healthcare_ca,Sharing content,"As a Content Amplifier, it is most aligned wit...",Sharing knowledge is power! As a radiology res...,agent5_healthcare_ca_10,agent6_healthcare_pc_1,agent6_healthcare_pc_1,healthcare,healthcare,True
686,10,agent61_music_ca,Sharing content,Sharing others' posts aligns with my role as a...,Delving into 19th-century European classical m...,agent61_music_ca_10,agent62_music_pc_6,agent62_music_pc_6,music,music,True


In [7]:
import re

# Definire i gruppi come liste
gruppoSO = []
gruppoOS = []
gruppoOE = []
gruppoBP = []
gruppoCA = []
gruppoPC = []
gruppoIE = []

# Popola le liste con gli agenti dei rispettivi gruppi
for row in df_log['Agent']:
    if re.search(r'_so$', row) and row not in gruppoSO:
        gruppoSO.append(row)
    elif re.search(r'_os$', row) and row not in gruppoOS:
        gruppoOS.append(row)
    elif re.search(r'_oe$', row) and row not in gruppoOE:
        gruppoOE.append(row)
    elif re.search(r'_bp$', row) and row not in gruppoBP:
        gruppoBP.append(row)
    elif re.search(r'_ca$', row) and row not in gruppoCA:
        gruppoCA.append(row)
    elif re.search(r'_pc$', row) and row not in gruppoPC:
        gruppoPC.append(row)
    elif re.search(r'_ie$', row) and row not in gruppoIE:
        gruppoIE.append(row)

# Funzione per assegnare il gruppo all'agente
def get_group(agent):
    if agent in gruppoSO:
        return 'SO'
    elif agent in gruppoOS:
        return 'OS'
    elif agent in gruppoOE:
        return 'OE'
    elif agent in gruppoBP:
        return 'BP'
    elif agent in gruppoCA:
        return 'CA'
    elif agent in gruppoPC:
        return 'PC'
    elif agent in gruppoIE:
        return 'IE'
    else:
        return 'Unknown'

# Aggiungi la colonna 'Group' nel DataFrame df_retweets
df_retweets['Group'] = df_retweets['Agent'].apply(get_group)

# Visualizza alcune righe per verificare
df_retweets[['Agent', 'Group', 'Agent_Topic', 'Original_Topic']].head()


C:\Users\Marco\AppData\Local\Temp\ipykernel_44700\879057173.py:49: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_retweets['Group'] = df_retweets['Agent'].apply(get_group)


,Agent,Group,Agent_Topic,Original_Topic
72,agent12_healthcare_ca,CA,healthcare,healthcare
76,agent16_healthcare_os,OS,healthcare,healthcare
78,agent18_healthcare_bp,BP,healthcare,healthcare
79,agent19_healthcare_ca,CA,healthcare,healthcare
84,agent23_technology_os,OS,technology,technology


In [8]:
# Aggiungi una colonna che verifica se i topic sono uguali
df_retweets['Same_Topic'] = df_retweets['Agent_Topic'] == df_retweets['Original_Topic']
df_retweets


C:\Users\Marco\AppData\Local\Temp\ipykernel_44700\2555855732.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_retweets['Same_Topic'] = df_retweets['Agent_Topic'] == df_retweets['Original_Topic']


,Iteration,Agent,Choice,Reason,Content,Content_ID,Original_Content_ID,Direct_Interaction_ID,Agent_Topic,Original_Topic,Same_Topic,Group
72,2,agent12_healthcare_ca,Sharing content,This choice aligns best with my role as a Cont...,As a pediatric healthcare provider specializin...,agent12_healthcare_ca_2,agent8_healthcare_so_1,agent8_healthcare_so_1,healthcare,healthcare,True,CA
76,2,agent16_healthcare_os,Sharing content,Aligns with my profile as an Occasional Sharer,"As a medical professional, I believe that shar...",agent16_healthcare_os_2,agent1_healthcare_so_1,agent1_healthcare_so_1,healthcare,healthcare,True,OS
78,2,agent18_healthcare_bp,Sharing content,"As a balanced participant, I need to contribut...",Did you know that sharing expertise and knowle...,agent18_healthcare_bp_2,agent15_healthcare_so_1,agent15_healthcare_so_1,healthcare,healthcare,True,BP
79,2,agent19_healthcare_ca,Sharing content,"As a Content Amplifier, my primary task is to ...",Exciting News! As we continue to push forward ...,agent19_healthcare_ca_2,agent21_healthcare_ie_1,agent21_healthcare_ie_1,healthcare,healthcare,True,CA
84,2,agent23_technology_os,Sharing content,Aligns with my profile as an Occasional Sharer,Let's spark a conversation about motor protect...,agent23_technology_os_2,agent27_technology_pc_1,agent27_technology_pc_1,technology,technology,True,OS
...,...,...,...,...,...,...,...,...,...,...,...,...
678,10,agent54_religion_ca,Sharing content,Given my behavioral profile as a Content Ampli...,Let's break down some common misconceptions ab...,agent54_religion_ca_10,agent55_religion_pc_8,agent44_religion_os_9,religion,religion,True,CA
682,10,agent58_music_os,Sharing content,"As an Occasional Sharer, it's best to stay mos...",As a proactive contributor and musicologist sp...,agent58_music_os_10,agent62_music_pc_3,agent60_music_bp_8,music,music,True,OS
684,10,agent5_healthcare_ca,Sharing content,"As a Content Amplifier, it is most aligned wit...",Sharing knowledge is power! As a radiology res...,agent5_healthcare_ca_10,agent6_healthcare_pc_1,agent6_healthcare_pc_1,healthcare,healthcare,True,CA
686,10,agent61_music_ca,Sharing content,Sharing others' posts aligns with my role as a...,Delving into 19th-century European classical m...,agent61_music_ca_10,agent62_music_pc_6,agent62_music_pc_6,music,music,True,CA


In [10]:
# Raggruppa per 'Group' (gruppo dell'agente) e calcola la percentuale di retweet sullo stesso topic
grouped = df_retweets.groupby('Group').agg(
    total_retweets=('Same_Topic', 'count'),  # Conta il totale di retweet per ogni gruppo
    same_topic_retweets=('Same_Topic', 'sum')  # Conta i retweet fatti sullo stesso topic
)

# Calcola la percentuale
grouped['Percentage_Same_Topic'] = (grouped['same_topic_retweets'] / grouped['total_retweets']) * 100

# Visualizza i risultati in Colab
display(grouped[['Percentage_Same_Topic']].reset_index())


,Group,Percentage_Same_Topic
0,BP,95.454545
1,CA,100.000000
2,OE,91.666667
3,OS,100.000000
4,SO,100.000000


Altre azioni

In [13]:
import pandas as pd

# Funzione per estrarre il topic da 'Agent' o 'Original_Content_ID'
def extract_topic(identifier):
    # Supponiamo che il topic sia la seconda parte dell'ID (dopo l'underscore)
    parts = identifier.split('_')
    return parts[1] if len(parts) > 1 else 'Unknown'

# Filtra tutte le interazioni (like, dislike, e commenti)
interaction_choices = ["Liking content", "Disliking content", "Commenting content"]
df_interactions = df_log[df_log['Choice'].isin(interaction_choices)].copy()

# Aggiungi una colonna per il gruppo dell'agente (usiamo la colonna 'Agent' per estrarre il gruppo)
def get_group(agent_name):
    if "_so" in agent_name:
        return "SO"
    elif "_os" in agent_name:
        return "OS"
    elif "_oe" in agent_name:
        return "OE"
    elif "_bp" in agent_name:
        return "BP"
    elif "_ca" in agent_name:
        return "CA"
    elif "_pc" in agent_name:
        return "PC"
    elif "_ie" in agent_name:
        return "IE"
    else:
        return "Unknown"

# Assegna il gruppo a ciascun agente
df_interactions['Group'] = df_interactions['Agent'].apply(get_group)

# Aggiungi una colonna con il topic dell'agente
df_interactions['Agent_Topic'] = df_interactions['Agent'].apply(extract_topic)

# Aggiungi una colonna con il topic del contenuto (Original_Content_ID)
df_interactions['Content_Topic'] = df_interactions['Original_Content_ID'].apply(extract_topic)

# Aggiungi una colonna che indica se l'interazione è sullo stesso topic
df_interactions['Same_Topic'] = df_interactions['Agent_Topic'] == df_interactions['Content_Topic']

# Raggruppa per 'Group' e 'Choice' (tipo di interazione) e calcola la percentuale di interazioni sullo stesso topic
grouped_interactions = df_interactions.groupby(['Group', 'Choice']).agg(
    total_interactions=('Same_Topic', 'count'),  # Conta il totale di interazioni per ogni gruppo e tipo
    same_topic_interactions=('Same_Topic', 'sum')  # Conta le interazioni sullo stesso topic
).reset_index()

# Calcola la percentuale di interazioni sullo stesso topic
grouped_interactions['Percentage_Same_Topic'] = (grouped_interactions['same_topic_interactions'] / grouped_interactions['total_interactions']) * 100

# Visualizza i risultati per le interazioni per ogni gruppo e tipo di azione
display(grouped_interactions[['Group', 'Choice', 'Percentage_Same_Topic']])


,Group,Choice,Percentage_Same_Topic
0,BP,Commenting content,100.000000
1,BP,Liking content,100.000000
2,CA,Commenting content,100.000000
3,CA,Liking content,100.000000
4,IE,Commenting content,97.727273
5,OE,Commenting content,93.750000
6,OE,Liking content,98.305085
7,OS,Disliking content,100.000000
8,OS,Liking content,100.000000
9,PC,Commenting content,100.000000


In [14]:
# Selezioniamo tutte le interazioni (like, dislike, comment)
interaction_choices = ["Liking content", "Disliking content", "Commenting content"]
df_interacting = df_log[df_log['Choice'].isin(interaction_choices)].copy()

# Aggiungi una colonna per il gruppo dell'agente
df_interacting['Group'] = df_interacting['Agent'].apply(get_group)

# Aggiungi una colonna con il topic dell'agente
df_interacting['Agent_Topic'] = df_interacting['Agent'].apply(extract_topic)

# Aggiungi una colonna con il topic del contenuto (Original_Content_ID)
df_interacting['Content_Topic'] = df_interacting['Original_Content_ID'].apply(extract_topic)

# Aggiungi una colonna che indica se l'interazione è sullo stesso topic
df_interacting['Same_Topic'] = df_interacting['Agent_Topic'] == df_interacting['Content_Topic']

# Raggruppa per 'Group' (gruppo dell'agente) e 'Choice' (interazione) e calcola la percentuale di interazioni sullo stesso topic
grouped_interacting = df_interacting.groupby(['Group']).agg(
    total_interactions=('Same_Topic', 'count'),  # Conta il totale delle interazioni
    same_topic_interactions=('Same_Topic', 'sum')  # Conta le interazioni sullo stesso topic
).reset_index()

# Calcola la percentuale di interazioni sullo stesso topic
grouped_interacting['Percentage_Same_Topic'] = (grouped_interacting['same_topic_interactions'] / grouped_interacting['total_interactions']) * 100

# Visualizza i risultati per le interazioni per ogni gruppo
display(grouped_interacting[['Group', 'Percentage_Same_Topic']])


,Group,Percentage_Same_Topic
0,BP,100.000000
1,CA,100.000000
2,IE,97.727273
3,OE,97.333333
4,OS,100.000000
5,PC,100.000000
6,SO,100.000000
